# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadFaizan0023/FlyRank_ML_internship_repo/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

 **My Rule:**<br> If CTR is low, engagement is low, low scroll event rate, position is greater than or equal to 10 and page is not updated for a quite long time than refresh page.

In [24]:
print("""
low ctr, low engagement rate, low scroll rate ,page position greater or far greater than 10, and page not updated for a while
""")


low ctr, low engagement rate, low scroll rate ,page position greater or far greater than 10, and page not updated for a while 



## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
import pandas as pd
import os, getpass
import duckdb

In [2]:
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')

dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


<h2>Taken sample data</h2>

In [52]:
clients_last_3m = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, gsc_impressions, gsc_clicks, gsc_avg_position,
    ga4_pageviews, ga4_sessions, ga4_engaged_sessions, scroll_events
    FROM {TABLES['fact_daily']}
    WHERE report_date >= '2026-04-01'
      AND report_date <= '2026-05-30'
      AND client_has_gsc == 'true'
      AND client_has_ga4 == 'true'
      AND gsc_data_available == 'true'
      AND ga4_data_available == 'true'
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [53]:
print(len(clients_last_3m))

1111616


In [54]:
clients_last_3m.columns

Index(['report_date', 'client_hash_id', 'content_hash_id', 'gsc_impressions',
       'gsc_clicks', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions',
       'ga4_engaged_sessions', 'scroll_events'],
      dtype='object')

In [55]:
clients_last_3m["gsc_impressions"].isna().sum()

np.int64(0)

In [19]:
reference_date = con.sql(f"""
    SELECT MAX(report_date) AS max_date
    FROM clients_last_3m
""").fetchone()[0]
print(f"Reference date set to: {reference_date}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Reference date set to: 2026-06-30


In [20]:
query = f"""
    SELECT
        f.*,
        d.content_updated_date,
        f.gsc_clicks / NULLIF(f.gsc_impressions, 0) AS ctr,
        f.ga4_engaged_sessions / NULLIF(f.ga4_sessions, 0) AS engagement_rate,
        f.scroll_events / NULLIF(f.ga4_pageviews, 0) AS scroll_rate,
        DATE_DIFF('day', CAST(d.content_updated_date AS DATE), DATE '{reference_date}') AS days_since_update
    FROM clients_last_3m f
    LEFT JOIN {TABLES['dim_content']} d
        ON f.content_hash_id = d.content_hash_id
"""

In [21]:
print(query)


    SELECT
        f.*,
        d.content_updated_date,
        f.gsc_clicks / NULLIF(f.gsc_impressions, 0) AS ctr,
        f.ga4_engaged_sessions / NULLIF(f.ga4_sessions, 0) AS engagement_rate,
        f.scroll_events / NULLIF(f.ga4_pageviews, 0) AS scroll_rate,
        DATE_DIFF('day', CAST(d.content_updated_date AS DATE), DATE '2026-06-30') AS days_since_update
    FROM clients_last_3m f
    LEFT JOIN read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet') d
        ON f.content_hash_id = d.content_hash_id



In [22]:
#output_path = os.makedirs('work/outputs', exist_ok=True)

# Execute query and save to parquet
con.sql(f"COPY ({query}) TO 'work/outputs/updated_features.csv' (FORMAT CSV)")
# Load into a dataframe for the next steps
#data = con.sql("SELECT * FROM 'updated_features.parquet'").df()
#print(f"Success! Data loaded. Shape: {data.shape}")
#display(data.head())


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [23]:
from google.colab import files
files.download('work/outputs/updated_features.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [56]:
dimf_content = con.sql(f"""
    SELECT content_hash_id, content_updated_date
    FROM {TABLES['dim_content']}
""").df()

In [57]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

clients_last_3m["ctr"] = clients_last_3m["gsc_clicks"] / clients_last_3m["gsc_impressions"]
clients_last_3m["engagement_rate"] = clients_last_3m["ga4_engaged_sessions"] / clients_last_3m["ga4_sessions"]
clients_last_3m["scroll_rate"] = clients_last_3m["scroll_events"] / clients_last_3m["ga4_pageviews"].replace(0, pd.NA)
clients_last_3m = clients_last_3m.merge(
    dimf_content,
    on="content_hash_id",
    how="left"          # keep all fact rows even if a page is missing from dim_content
)
reference_date = clients_last_3m["report_date"].max()
clients_last_3m["days_since_update"] = (reference_date - clients_last_3m["content_updated_date"]).dt.days


In [58]:
clients_last_3m.columns

Index(['report_date', 'client_hash_id', 'content_hash_id', 'gsc_impressions',
       'gsc_clicks', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions',
       'ga4_engaged_sessions', 'scroll_events', 'ctr', 'engagement_rate',
       'scroll_rate', 'content_updated_date', 'days_since_update'],
      dtype='object')

In [59]:
clients_last_3m = clients_last_3m.drop(columns=['content_updated_date','report_date', 'client_hash_id', 'gsc_impressions',
       'gsc_clicks','ga4_pageviews', 'ga4_sessions',
       'ga4_engaged_sessions', 'scroll_events'])

In [60]:
clients_last_3m.columns

Index(['content_hash_id', 'gsc_avg_position', 'ctr', 'engagement_rate',
       'scroll_rate', 'days_since_update'],
      dtype='object')

In [61]:
clients_last_3m["scroll_rate"].value_counts()

,count
scroll_rate,
0.0,927122
1.0,59581
0.5,42139
0.25,14407
0.333333,14260
...,...
0.348837,1
0.082873,1
0.019504,1


In [ ]:
ctr_threshold = 0.2
engagement_rate_threshold = 0.3
scroll_rate_threshold = 300

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.